# Earnings-Event Intelligence — Colab Quickstart

End-to-end tour of the `earnings-ml` research pipeline on **synthetic data with a planted
signal**: event alignment → point-in-time features → leakage checks → walk-forward
training → portfolio backtest with transaction costs → negative control.

> **Everything you will see is SYNTHETIC.** The default provider is deterministic mock
> data; the numbers validate the *pipeline and methodology*, not US equity markets.
> See `docs/ADR.md` (ADR-001) in the repo.

**Runtime:** ~3–5 minutes on a Colab CPU. No GPU needed.

**What this notebook demonstrates**
1. Effective-trading-date alignment (BMO / AMC / weekend rules) on an exchange calendar
2. Point-in-time feature stamps (`feature_available_at < order_submission_cutoff`, strict)
3. Expanding-window walk-forward evaluation (no shuffling — ever)
4. Portfolio backtest where the long-short leg pays costs on **both** sides
5. Negative control: with the planted signal removed (γ = 0), no alpha may appear

## 0. Setup

If you are on Colab, either paste your git URL below **or** upload the project folder to
`/content` (Files panel → upload, or mount Drive). Locally, just run the notebook from the
repository root.

In [ ]:
# SKIP_LOCAL  # environment bootstrap — exercised manually, not by the local test harness
import os
import subprocess
import sys

REPO_URL = ""  # e.g. "https://github.com/<you>/earning-event-intelligence.git" (leave "" to upload manually)
PROJECT_DIR = "/content/earning-event-intelligence"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "numpy", "scikit-learn", "statsmodels", "lightgbm",
                    "pyarrow", "pandera", "exchange_calendars", "click", "pyyaml",
                    "matplotlib", "jinja2"], check=True)
    if REPO_URL and not os.path.isdir(PROJECT_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, PROJECT_DIR], check=True)
    if not os.path.isdir(PROJECT_DIR):
        raise FileNotFoundError(
            f"Project not found at {PROJECT_DIR}. Upload the repository folder "
            "(Files panel) or set REPO_URL above."
        )
    os.chdir(PROJECT_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    # local: walk up to the repo root (the directory containing pyproject.toml)
    while not os.path.exists("pyproject.toml") and os.getcwd() != "/":
        os.chdir("..")
    assert os.path.exists("pyproject.toml"), "run from inside the repository"

PROJECT_DIR = os.getcwd()
print("Colab:" , IN_COLAB, "| project:", PROJECT_DIR)

## 1. Configuration

We shrink the universe (30 stocks) so the notebook runs in minutes. Everything else —
seed, calendar, planted-signal parameters — comes from `configs/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from earnings_ml.backtest.portfolio import run_backtest
from earnings_ml.config import load_config
from earnings_ml.event_alignment.master import build_event_master, validate_effective_dates
from earnings_ml.evaluation.walkforward import run_walkforward
from earnings_ml.features.structured import build_structured_features
from earnings_ml.ingestion.mock import MockProvider
from earnings_ml.labels.car import attach_latent_labels, build_labels
from earnings_ml.models.zoo import LightGBMReg, LinearReg, make_baselines
from earnings_ml.utils import seed_everything
from earnings_ml.validation.invariants import check_feature_availability

cfg = load_config("configs")
cfg.base["universe"]["n_stocks"] = 30            # smaller universe -> faster notebook
seed_everything(cfg.seed)

print("provider:", cfg.provider, "| seed:", cfg.seed, "| tz:", cfg.tz)
print("planted signal:", cfg.base["signal"]["gamma_base"],
      "(reaction CAR strength, decays",
      cfg.base["signal"]["gamma_decay_per_year"], "per year)")

## 2. Ingest the synthetic universe & align events

Every announcement is mapped to its **effective trading date** using the XNYS calendar
(no calendar-day arithmetic):
- **BMO** (before open) → same session
- **AMC** (after close) → next session
- weekend/holiday → next session
- intraday / unknown timing → **excluded** from the primary sample

In [ ]:
out = MockProvider(cfg).fetch()
master = build_event_master(out.events, cfg)

n_primary = int((master["exclusion_reason"].fillna("") == "").sum())
print(f"events: {len(master)} | primary: {n_primary} | "
      f"prices rows: {len(out.prices):,} | synthetic: {out.is_synthetic}")

align = validate_effective_dates(master)
failures = int((~align["is_session"]).sum() + (~align["amc_next_session_ok"]).sum()
               + (~align["bmo_same_session_ok"]).sum()
               + (~align["intraday_unknown_excluded_ok"]).sum())
print("alignment failures:", failures)
assert failures == 0

master[["ticker", "timing_class", "announcement_timestamp",
        "effective_trading_date", "exclusion_reason"]].head(8)

## 3. Point-in-time features & the leakage invariant

Every feature row carries `feature_available_at` — when the value was truly observable:
- **surprise** features: at the announcement (actuals are not public at estimate time)
- **market-context** features: at the prior session's close (16:05 ET)
- **cross-sectional** features: effective date 00:00 ET

A merged row is available at the **max** of its components and must be **strictly before**
the 09:25 ET order-submission cutoff. This is the repo's core anti-leakage invariant.

In [ ]:
feats = build_structured_features(master, out.prices, out.benchmark_prices, cfg)

bad = check_feature_availability(feats, strict=True)
print("PIT violations:", len(bad), f"({len(feats)} rows checked)")
assert len(bad) == 0

demo = feats.merge(master[["event_id", "timing_class"]], on="event_id")
demo = demo[["event_id", "timing_class", "announcement_timestamp",
             "feature_available_at", "order_submission_cutoff"]].copy()
demo["margin_minutes"] = (
    pd.to_datetime(demo["order_submission_cutoff"], utc=True)
    - pd.to_datetime(demo["feature_available_at"], utc=True)
).dt.total_seconds() / 60
demo.head(8)

## 4. Labels

- **Reaction CAR**: entry-session open → close (the tradable reaction window)
- **Drift CAR (20)**: reaction-close → +20 sessions (never overlaps the reaction window)

On the synthetic path, labels are then overridden with the provider's **planted** values
(`attach_latent_labels`) — the pipeline must be able to *recover* the planted signal and
must find *nothing* when it is removed (section 7).

In [ ]:
labels = build_labels(master, out.prices, out.benchmark_prices, out.sector_prices, cfg)
labels = attach_latent_labels(master, labels)  # synthetic path only (documented in report)
labels[["reaction_car", "drift_car_20", "realized_vol_20"]].describe().round(4)

## 5. Walk-forward evaluation

Expanding window: train 2015–2018, then test folds roll forward one year at a time
(2020–2025). **No random splits, no shuffling** — an AST-level guard in the evaluation
module forbids `train_test_split`/`KFold`/`ShuffleSplit`, and preprocessing is fit on the
training fold only. (~1–2 min)

In [ ]:
models = make_baselines() + [LinearReg(), LightGBMReg()]
res = run_walkforward(feats, labels, cfg, models=models, target="reaction_car")

pooled = pd.DataFrame(res["pooled_metrics"]).T
cols = ["n_total", "mae", "pearson", "pearson_ci_low", "pearson_ci_high",
        "top_minus_bottom", "roc_auc"]
pooled[cols].sort_values("pearson", ascending=False).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# information coefficient by model
pooled["pearson"].sort_values().plot.barh(ax=axes[0], color="#003a70")
axes[0].set_title("Pooled Pearson IC by model (reaction CAR)")
axes[0].axvline(0, color="k", lw=0.8)

# signal decay across walk-forward folds (the planted gamma decays over calendar time)
fm = res["fold_metrics"]
best = pooled["pearson"].idxmax()
per_fold = fm[fm["model"] == best].set_index("fold")["pearson"]
per_fold.plot.bar(ax=axes[1], color="#b35900")
axes[1].set_title(f"Per-fold IC — {best} (planted signal decays over time)")
axes[1].axhline(0, color="k", lw=0.8)

plt.tight_layout(); plt.show()

## 6. Economic backtest with a cost sweep

Top-minus-bottom **quintile** portfolios, formed **within each test fold** (no cross-fold
lookahead in ranking). Costs are a round trip and are charged on **both** legs of the
long-short — so the L/S line must slope down; where it crosses zero is the breakeven
cost. On synthetic data without microstructure this is a *plumbing* demonstration, not an
economic claim (ADR-005).

In [ ]:
preds = res["predictions"]
mp = preds[preds["model"] == "lightgbm"]
bt = run_backtest(master, mp, out.prices, cfg, horizon="drift")

sweep = pd.DataFrame(bt["cost_sweep"]).T
sweep.index = sweep.index.astype(int)
sweep = sweep.sort_index()

ax = sweep[["long_only_net", "long_short_net"]].plot(marker="o", figsize=(7, 4))
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("round-trip cost (bps)"); ax.set_ylabel("mean net return per event")
ax.set_title("Drift-horizon portfolios vs transaction cost (lightgbm)")
plt.show()

sweep[["n_long", "long_only_net", "short_only_net", "long_short_net"]].round(5)

## 7. Negative control — γ = 0 must yield no alpha

The single most important integrity check for a synthetic-data project. We regenerate the
universe with the planted signal **removed** (`gamma_base = 0`) and re-run the identical
pipeline. If any model still shows a large information coefficient, the pipeline is
manufacturing alpha from noise — i.e. there is leakage somewhere. (The repo enforces this
in CI: `tests/test_negative_control.py` requires best |IC| < 0.15.) (~1–2 min)

In [ ]:
import copy

cfg_nc = copy.deepcopy(cfg)
cfg_nc.base["signal"]["gamma_base"] = 0.0
cfg_nc.base["signal"]["drift_multiplier"] = 0.0

out_nc = MockProvider(cfg_nc).fetch()
master_nc = build_event_master(out_nc.events, cfg_nc)
# features are identical across the two runs (gamma only moves the planted labels)
labels_nc = attach_latent_labels(
    master_nc, build_labels(master_nc, out_nc.prices, out_nc.benchmark_prices,
                            out_nc.sector_prices, cfg_nc)
)
res_nc = run_walkforward(feats, labels_nc, cfg_nc, models=models, target="reaction_car")

cmp = pd.DataFrame({
    "IC with planted signal": pd.DataFrame(res["pooled_metrics"]).T["pearson"],
    "IC with gamma = 0": pd.DataFrame(res_nc["pooled_metrics"]).T["pearson"],
}).sort_values("IC with planted signal", ascending=False)

best_nc = cmp["IC with gamma = 0"].abs().max()
print(f"best |IC| under negative control: {best_nc:.4f}  (must be < 0.15)")
assert best_nc < 0.15, "pipeline manufactures alpha from noise — leakage!"

cmp.plot.bar(figsize=(10, 4)); plt.axhline(0, color="k", lw=0.8)
plt.title("Information coefficient: planted signal vs negative control")
plt.tight_layout(); plt.show()
cmp.round(4)

## 8. Where to go next

| Artifact | Location |
|---|---|
| Full HTML research report | `artifacts/report/report.html` (after `earnings-ml run-all`) |
| Design decisions | `docs/ADR.md` |
| Reproduction steps | `docs/REPRODUCING.md` |
| Leakage / negative-control tests | `tests/test_leakage.py`, `tests/test_negative_control.py` |

**To go further**
- Run the full 100-stock pipeline: `earnings-ml run-all` (writes parquet + JSON artifacts)
- Turn on text features: `configs/features.yaml → text.enabled: true` (see ADR-002 for why
  they are OFF by default)
- Real prices smoke path: `configs/base.yaml → provider: edgar_yf` — note events remain
  synthetic there, so outputs stay labeled SYNTHETIC (ADR-003)

*Reminder: no number in this notebook is evidence of alpha in US equities.*